<a href="https://colab.research.google.com/github/Anemodude/sustainability-index-M608-project-/blob/main/M608_sustainability_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install dash


In [6]:
import pandas as pd
import numpy as np
import requests
import plotly.graph_objects as go
from dash import Dash, dcc, html

POTSDAM_LAT = 52.39
POTSDAM_LON = 13.06
timezone = "Europe/Berlin"
weather_url = (
    "https://api.open-meteo.com/v1/forecast?"
    f"latitude={POTSDAM_LAT}&longitude={POTSDAM_LON}&hourly="
    "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,"
    "uv_index,cloudcover,surface_pressure"
    "&timezone=Europe%2FBerlin"
)

weather_resp = requests.get(weather_url).json()

df = pd.DataFrame({
    "timestamp": pd.to_datetime(weather_resp["hourly"]["time"]),
    "air_temp": weather_resp["hourly"]["temperature_2m"],
    "humidity": weather_resp["hourly"]["relative_humidity_2m"],
    "wind_speed": weather_resp["hourly"]["wind_speed_10m"],
    "rain": weather_resp["hourly"]["precipitation"],
    "uv_index": weather_resp["hourly"]["uv_index"],
    "cloudcover": weather_resp["hourly"]["cloudcover"],
    "surface_pressure": weather_resp["hourly"]["surface_pressure"]
})

df["timestamp"] = df["timestamp"].dt.tz_localize(None)


def normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

components = [
    normalize(24 - abs(df["air_temp"] - 21)),
    normalize(60 - abs(df["humidity"] - 50)),
    normalize(5 - abs(df["wind_speed"] - 3)),
    normalize(3 - df["rain"]),
    normalize(10 - df["uv_index"]),
    normalize(50 - abs(df["cloudcover"] - 50)),
    normalize(1013 - abs(df["surface_pressure"] - 1013))
]

df["SI"] = sum(components) / len(components)
latest = df.iloc[0]

pie_values = [comp.iloc[0] for comp in components]
pie_labels = ["Temperature", "Humidity", "Wind", "Rain", "UV Index", "Cloud Cover", "Pressure"]


chart_layout = dict(
    showlegend=False,
    margin=dict(l=30, r=15, t=35, b=25),
    font=dict(size=10, family="Arial", color="#e0e0e0"),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    xaxis=dict(gridcolor="#2c2c2c", zeroline=False, color="#cfcfcf"),
    yaxis=dict(gridcolor="#2c2c2c", zeroline=False, color="#cfcfcf")
)


fig_gauge = go.Figure(go.Indicator(
    mode="gauge+number",
    value=float(latest["SI"] * 100),
    gauge={
        "axis": {"range": [0, 100], "tickwidth": 1, "tickcolor": "white","tickfont":{"color":"white","weight":"bold"}},
        "bar": {"color": "#808000"},
        "steps": [
            {"range": [0, 40], "color": "red"},
            {"range": [40, 70], "color": "yellow"},
            {"range": [70, 100], "color": "green"}
        ],
    },
    number={"font": {"size": 34, "weight": "bold", "color":"#81c784"}, "suffix": "%"}
))
fig_gauge.update_layout(
    title={"text": "Sustainability Index (SI)", "font": {"size": 12, "weight": "bold","color":"#81c784"}},
    margin=dict(l=20, r=20, t=45, b=15),
    height=180,
    paper_bgcolor="rgba(0,0,0,0)"
)


fig_pie = go.Figure(go.Pie(
    labels=pie_labels,
    values=pie_values,
    textinfo="percent",
    textposition="inside",
    hole=0.4,
    marker=dict(colors=["red", "#1a98a6", "white", "purple",
                        "orange", "gray", "#654321"])
))
fig_pie.update_layout(
    title={"text": "KPI Contribution Balance", "font": {"size": 12, "weight": "bold","color":"#81c784"}},
    margin=dict(l=10, r=10, t=45, b=15),
    height=180,
    showlegend=True,
    legend=dict(font=dict(size=8,color="white",weight="bold"), orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02),
    paper_bgcolor="rgba(0,0,0,0)"
)


line_configs = [
    ("air_temp", "Air Temperature (°C)", "red"),
    ("humidity", "Humidity (%)", "#1a98a6"),
    ("wind_speed", "Wind Speed (m/s)", "White"),
    ("rain", "Rainfall (mm)", "purple")
]

line_charts = []
for col, title, color in line_configs:
    fig = go.Figure(go.Scatter(
        x=df["timestamp"], y=df[col],
        mode="lines", line=dict(color=color, width=2)
    ))
    fig.update_layout(title={"text": title, "font": {"size": 12, "weight": "bold","color":"#81c784"}}, height=180, **chart_layout)
    line_charts.append(fig)


bar_configs = [
    ("uv_index", "UV Index", "orange"),
    ("cloudcover", "Cloud Cover (%)", "gray"),
    ("surface_pressure", "Pressure (hPa)", "#654321")
]

bar_charts = []
for col, title, color in bar_configs:
    fig = go.Figure(go.Bar(x=df["timestamp"], y=df[col], marker_color=color))

    if col == "surface_pressure":
        fig.update_layout(yaxis=dict(range=[df[col].min() - 2, df[col].max() + 2]))

    fig.update_layout(title={"text": title, "font": {"size": 12, "weight": "bold","color":"#81c784"}}, height=180, **chart_layout)
    bar_charts.append(fig)


app = Dash(__name__,title="Potsdam Sustainability Dashboard")

card_style = {
    'flex': 1,
    'backgroundColor': '#1e1e1e',
    'borderRadius': '6px',
    'boxShadow': '0 1px 3px rgba(0,0,0,0.4)',
    'padding': '5px',
    'boxSizing': 'border-box'
}

app.layout = html.Div(
    style={
        'fontFamily': 'Arial, sans-serif',
        'backgroundColor': '#000000',
        'height': '100vh',
        'padding': '12px',
        'display': 'flex',
        'flexDirection': 'column',
        'gap': '12px',
        'overflow': 'hidden',
        'boxSizing': 'border-box'
    },
    children=[

        html.Div("Potsdam Sustainability Dashboard",
                 style={'textAlign': 'center', 'fontSize': '26px',
                        'fontWeight': 'bold', 'color': '#3EB489',
                        'marginBottom': '10px'}),



        html.Div(
            style={'display': 'flex', 'gap': '12px', 'height': '190px'},
            children=[
                html.Div(dcc.Graph(figure=fig_gauge, style={'height': '100%'}), style=card_style),
                html.Div(dcc.Graph(figure=fig_pie, style={'height': '100%'}), style=card_style)
            ]
        ),


        html.Div(
            style={'display': 'flex', 'gap': '12px', 'height': '190px'},
            children=[
                html.Div(dcc.Graph(figure=line_charts[0], style={'height': '100%'}), style=card_style),
                html.Div(dcc.Graph(figure=line_charts[1], style={'height': '100%'}), style=card_style),
                html.Div(dcc.Graph(figure=line_charts[2], style={'height': '100%'}), style=card_style),
                html.Div(dcc.Graph(figure=line_charts[3], style={'height': '100%'}), style=card_style)
            ]
        ),


        html.Div(
            style={'display': 'flex', 'gap': '12px', 'height': '190px'},
            children=[
                html.Div(dcc.Graph(figure=bar_charts[0], style={'height': '100%'}), style=card_style),
                html.Div(dcc.Graph(figure=bar_charts[1], style={'height': '100%'}), style=card_style),
                html.Div(dcc.Graph(figure=bar_charts[2], style={'height': '100%'}), style=card_style)
            ]
        )
    ]
)


#app.run(jupyter_mode="inline")

app.run(jupyter_mode="external")



Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>